<a href="https://colab.research.google.com/github/Naganarthanan/RiceGuard-DL/blob/main/RiceGuard_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os, getpass
os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("Kaggle token paste pannunga: ")
!pip install -q -U kaggle

Kaggle token paste pannunga: ··········
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 5.0 MB/s eta 0:00:00


In [4]:
!kaggle datasets download -d anshulm257/rice-disease-dataset
!unzip -q rice-disease-dataset.zip -d rice_disease
!ls rice_disease

Dataset URL: https://www.kaggle.com/datasets/anshulm257/rice-disease-dataset
License(s): unknown
100% 0.99G/0.99G [00:09<00:00, 109MB/s] 

Rice_Leaf_AUG


In [5]:
!ls /content/rice_disease/Rice_Leaf_AUG

'Bacterial Leaf Blight'  'Healthy Rice Leaf'  'Leaf scald'
'Brown Spot'		 'Leaf Blast'	      'Sheath Blight'


In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_dir = '/content/rice_disease/Rice_Leaf_AUG'

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2
)

train_data = datagen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training'
)
val_data = datagen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation'
)

print("Classes found:", train_data.class_indices)

Found 3066 images belonging to 6 classes.
Found 763 images belonging to 6 classes.
Classes found: {'Bacterial Leaf Blight': 0, 'Brown Spot': 1, 'Healthy Rice Leaf': 2, 'Leaf Blast': 3, 'Leaf scald': 4, 'Sheath Blight': 5}


In [7]:
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

num_classes = train_data.num_classes

custom_cnn = models.Sequential([
    layers.Input(shape=(224,224,3)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

custom_cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = custom_cnn.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=[early_stop]
)

Epoch 1/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 82s 768ms/step - accuracy: 0.2707 - loss: 1.8479 - val_accuracy: 0.4010 - val_loss: 1.4383
Epoch 2/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 66s 681ms/step - accuracy: 0.4188 - loss: 1.4164 - val_accuracy: 0.4548 - val_loss: 1.3108
Epoch 3/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 65s 677ms/step - accuracy: 0.4889 - loss: 1.2716 - val_accuracy: 0.4718 - val_loss: 1.2789
Epoch 4/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 65s 679ms/step - accuracy: 0.5170 - loss: 1.2075 - val_accuracy: 0.4862 - val_loss: 1.2305
Epoch 5/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 66s 691ms/step - accuracy: 0.5538 - loss: 1.1223 - val_accuracy: 0.5269 - val_loss: 1.1406
Epoch 6/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 68s 711ms/step - accuracy: 0.5724 - loss: 1.0906 - val_accuracy: 0.5714 - val_loss: 1.0986
Epoch 7/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 65s 685ms/step - accuracy: 0.5656 - loss: 1.0940 - val_accuracy: 0.5819 - val_loss: 1.0598
Epoch 8/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 66s 685ms/step - accuracy: 0.6047 - loss: 1.0399 - val_accu